# Load the images + plant segmentations and overlay them

One repo, two configs:

| config | contents | download |
| --- | --- | --- |
| default | photos + metadata | ~1.3 GB |
| `segmentations` | same rows + plant masks, no images | ~32 MB |

Only rows **100–999** have masks (up to 9 plants per frame). Mask coordinates are in the
**undistorted 1920x1920** frame, so the image has to be de-fished before overlaying.

You need the Hugging Face token from Discord and access to the private dataset.

In [ ]:
!uv pip install datasets huggingface_hub pillow defisheye matplotlib pandas "pycocotools>=2.0.10"

# OR
# !uv pip install datasets huggingface_hub pillow defisheye matplotlib pandas "pycocotools>=2.0.10"

!git clone https://github.com/regularpooria/CanopyMap.git

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import os

hf_token = userdata.get("HF_KEY")
login(token=hf_token)  # paste the token from discord

### Load both configs

`load_dataset` returns a `DatasetDict`; we keep the `train` split. The segmentations
config is small, so it loads in seconds. The image config is cached after the first run.

In [ ]:
from datasets import load_dataset

DATASET = "regularpooria/canopymap_plants"

images = load_dataset(DATASET, split="train")  # photos
segs = load_dataset(DATASET, "segmentations", split="train")  # masks, row i == images row i

print(images)
print(segs)

### What's in a segmentation row

| column | meaning |
| --- | --- |
| `frame_idx` | row index in the image config (0–3162) |
| `seg_n_plants` | number of tracked plants in this frame |
| `seg_ids` | plant ids, stable across frames |
| `seg_bboxes_xyxy` | boxes `[x1, y1, x2, y2]` in undistorted pixels |
| `seg_areas` | mask area in pixels |
| `seg_rle` | COCO compressed RLE per plant |

All lists are aligned positionally. The `ph_*` columns are plant-holder detections, not these masks.

In [ ]:
row = segs[900]
print({k: (v if k != "seg_rle" else f"<{len(v)} masks>") for k, v in row.items()})

In [ ]:
import sys
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from pycocotools import mask as mask_utils

from CanopyMap.src.utils.distortion import de_fish_eye_image


def undistort(image):
    """De-fish a stored photo the same way the tracker did."""
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "frame.jpg"
        image.convert("RGB").save(path, quality=95)
        bgr = de_fish_eye_image(str(path))
    return bgr[..., ::-1]  # BGR -> RGB


def color_for(pid):
    """A fixed color per plant id."""
    r, g, b, _ = plt.cm.tab10(pid % 10)
    return np.uint8([r * 255, g * 255, b * 255])

In [ ]:
def show(frame_idx, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 10))

    seg = segs[frame_idx]
    frame = undistort(images[frame_idx]["image"])

    overlay = np.zeros((*frame.shape[:2], 4), dtype=np.uint8)
    for pid, (x1, y1, _, _), rle in zip(seg["seg_ids"], seg["seg_bboxes_xyxy"], seg["seg_rle"]):
        mask = mask_utils.decode({"size": [1920, 1920], "counts": rle}).astype(bool)
        overlay[mask] = (*color_for(pid), 130)
        ax.text(x1 + 6, y1 + 20, str(pid), color="white", fontsize=11, weight="bold")

    ax.imshow(frame)
    ax.imshow(overlay)
    ax.set_title(
        f"frame {frame_idx} | {seg['channel']} | {seg['date']} {seg['time']} "
        f"| {seg['seg_n_plants']} plants",
        fontsize=10,

    )
    ax.axis("off")

In [ ]:
show(900)
plt.show()

### Why de-fish first

The stored photos are raw fisheye (2560x1920); the masks live in the undistorted 1920x1920
frame. Drawing the mask on the raw photo makes it look shifted, which is expected.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].imshow(images[100]["image"])
axes[0].set_title("raw fisheye")
axes[0].axis("off")

show(100, ax=axes[1])
axes[1].set_title("undistorted + masks")
plt.tight_layout()
plt.show()

# PUT YOUR CODE HERE

## Probably some code to use the segmentation data to calculate area for each plant

In [ ]:
# Append plant segmentation data into a df
import pandas as pd

rows = []
for i in range(100, 1000):          # only these rows have masks
    s = segs[i]
    for pid, area in zip(s["seg_ids"], s["seg_areas"]):
        rows.append({
            "frame_idx": i,
            "plant_id": pid,
            "area_px": area,
            "timestamp": pd.to_datetime(f"{s['date']} {s['time']}"),
        })

df = pd.DataFrame(rows)
df.head()
# print(len(df))  # Expect (#frames)*(#plants/frame) = (1000-100)*9 = 900*9 = 8100 segmentations

## Some code to plot the area over time for each plant

### Master Plot: Big'ol plot with all plants over entire range (100,999)

In [ ]:
# import matplotlib.pyplot as plt   # already imported from above
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(14, 6)) # idk size can change

for pid, g in df.groupby("plant_id"):   # Group by plant_id
    g = g.sort_values("timestamp")      # Sort each plant_id group by time to get growth over time
    ax.plot(g["timestamp"], g["area_px"], label=f"plant {pid}", linewidth=1.5)
    # print(g)  # sanity check

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))   # Set x-axis format

ax.set_xlabel("Time")
ax.set_ylabel("Mask area (px, undistorted frame)")
ax.set_title("Plant area over time")
ax.legend(title="Plant ID", ncol=3)
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.show()

### Shortened range for various smaller intervals

In [ ]:
df_small = df.iloc[:828]    # Change to interval we want to specifically isolate

fig, ax = plt.subplots(figsize=(14, 6))

for pid, g in df_small.groupby("plant_id"):
    g = g.sort_values("timestamp")
    ax.plot(g["timestamp"], g["area_px"], label=f"plant {pid}", linewidth=1.5)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))   # Set the format for x-axis (for clarity)

ax.set_xlabel("Time")
ax.set_ylabel("Mask area (px, undistorted frame)")
ax.set_title("Plant area over time (first day)")
ax.legend(title="Plant ID", ncol=3)
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.show()

That interval between 05-18 00:00 and 05-18 07:00 looks suspicious. Seems to consistently occur in large plot too. Check difference between sorted list of recorded timestamps.

In [ ]:
times = df_small["timestamp"].drop_duplicates().sort_values()
print(times.diff().value_counts())  # There is a large difference. The gap in data must've been from the greeness filtering, just something to note. idk.

If you want, instead of area, you can count the green pixels within the segmentation and plot that instead, keep both plots, it would be cool to see the difference :(

# Green Pixel Ver
## Calculate Green pixels

Step 1: Adjust green pixel threshold

In [ ]:
# This function will check each pixel in the frame and return if the pixel is above a greeness threshold
# Either way, main goal is to ensure that plant pixels in the segmentation masks are green, and the black
# background is not counted, so it does not need to be 100% accurate.
def green_mask(frame_rgb, thresh=0.05):   # mess around with the threshold
    f = frame_rgb.astype(np.float32) / 255
    r, g, b = f[..., 0], f[..., 1], f[..., 2]
    return (2 * g - r - b) > thresh

# Threshold check: Test on a random frame
frame = undistort(images[100]["image"])
green = green_mask(frame)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(frame);  axes[0].set_title("undistorted")
axes[1].imshow(frame);  axes[1].imshow(green, alpha=0.6, cmap="autumn")
axes[1].set_title("Green = True (is green), Red = False (not green)")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

Note: Looks like this particular way of calculating green pixels is struggling with detecting light (white) pixels from the bright lamp

Step 2: Count green pixels per plant segmentation mask

In [ ]:
frame_idx = 100   # Test random plant
s = segs[frame_idx]
frame = undistort(images[frame_idx]["image"])
green = green_mask(frame, thresh=0.05)

overlay = np.zeros((*frame.shape[:2], 4), dtype=np.uint8)

for pid, area, rle in zip(s["seg_ids"], s["seg_areas"], s["seg_rle"]):
    m = mask_utils.decode({"size": [1920, 1920], "counts": rle}).astype(bool)
    counted = m & green          # inside the mask AND green
    ignored = m & ~green         # inside the mask but NOT green

    overlay[counted] = (0, 255, 0, 140)     # green = counted
    overlay[ignored] = (255, 0, 0, 140)     # red = in mask, not counted

    print(f"plant {pid}: mask area = {area}, green = {counted.sum()}, "
          f"fraction = {counted.sum() / area:.2f}")

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(frame)
ax.imshow(overlay)
ax.set_title(f"frame {frame_idx}: green = counted, red = in mask but not green")
ax.axis("off")
plt.show()

Note: Looks solid, move onto loop and plotting

## Plot Green Pixels over time

Step 1: Perform green pixel counting on dataset (do subset first for proof of concept)

In [ ]:
from tqdm.auto import tqdm    # just show progress cause cool

rows = []
for i in tqdm(range(100, 200)):          # start with the 100-frame subset
    s = segs[i]
    frame = undistort(images[i]["image"])   # Count on undistorted frame
    green = green_mask(frame)   # Count the GREEEEEEEEEEN

    for pid, area, rle in zip(s["seg_ids"], s["seg_areas"], s["seg_rle"]):
        m = mask_utils.decode({"size": [1920, 1920], "counts": rle}).astype(bool)
        rows.append({
            "frame_idx": i,
            "plant_id": pid,
            "area_px": area,
            "green_px": int((m & green).sum()),
            "timestamp": pd.to_datetime(f"{s['date']} {s['time']}"),
        })

df_green = pd.DataFrame(rows)
df_green["green_frac"] = df_green["green_px"] / df_green["area_px"]
df_green.to_csv("plant_green_100_199.csv", index=False)   # cache it
df_green.head()

Step 2: Plot original segmentation area vs green pixel area and compare

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharex=True)

for pid, g in df_green.groupby("plant_id"):
    g = g.sort_values("timestamp")
    axes[0].plot(g["timestamp"], g["area_px"], label=f"plant {pid}")
    axes[1].plot(g["timestamp"], g["green_px"], label=f"plant {pid}")

axes[0].set_title("Mask area (px)")
axes[1].set_title("Green pixels (px)")
for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    ax.set_xlabel("Time")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Pixels")
axes[1].legend(title="Plant ID", ncol=3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

This was 99% Claude, 1% prompting by 2 of my 5 braincells :)

Next Steps:
- Adjust/Find better green pixel counter algorithm
- Add a function that checks that plant size does not decrease over time